# HMR-GNN — Run experiments on a free Colab GPU

**Heterophily-Aware Multi-Relational GNN for Robust Fake Account Detection.**

This notebook clones the repo (code + MGTAB data), installs dependencies, runs the
full experiment suite (baselines + hyperparameter tuning + ablation) on the **bot**
(fake-account) task across 5 seeds, and downloads all paper tables.

### Before you start
1. Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**, then Save.
2. Run each cell top to bottom (Shift+Enter).

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Clone the repository (code + data)

In [ ]:
import os
if not os.path.isdir('/content/HMR-GNN'):
    !git clone https://github.com/Vincentfernandes17/HMR-GNN.git /content/HMR-GNN
%cd /content/HMR-GNN
!git pull
# sanity check that the data shipped with the repo
assert os.path.exists('data/MGTAB/features.pt'), 'MGTAB data missing from repo!'
print('Repo ready. Data files:', os.listdir('data/MGTAB'))

## 3. Install dependencies
Colab already ships a GPU build of PyTorch, so we only install the rest to avoid
downgrading torch to a CPU build.

In [ ]:
!pip install -q "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" "numpy>=1.24"
import sklearn, scipy, pandas, numpy
print('sklearn', sklearn.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__)

## 4. Quick smoke test (about 1 minute)
Runs 3 models for 2 epochs to confirm everything works before the long run.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py --mode smoke --task bot --epochs 2 --patience 2 --quiet
print('\nSmoke test finished OK.')

## 5. Full experiment suite (resilient to disconnects)

Free Colab sessions can disconnect during long runs. To protect your progress, this
cell:

1. **Saves all results to your Google Drive** (`MyDrive/HMR-GNN-results`) so nothing is
   lost if the session drops.
2. Runs the suite in **three phases** — baselines, ablation, hyperparameter tuning —
   each writing its tables as soon as it finishes.
3. **Skips any phase that already completed**, so if you get disconnected you can simply
   re-run this cell and it continues where it left off.

On a T4 GPU the whole thing is roughly **1-3 hours**. You can also run it in chunks across
several sessions thanks to the skip logic.

*Tip: if you are short on time, change `SEEDS` to `42,52,62` and `TRIALS` to `6` below.*

In [ ]:
import os

# --- Persist results to Google Drive so disconnects don't lose progress ---
from google.colab import drive
drive.mount('/content/drive')
OUTDIR = '/content/drive/MyDrive/HMR-GNN-results'
os.makedirs(OUTDIR, exist_ok=True)
print('Results will be saved to:', OUTDIR)

# --- Experiment settings (lower these if you are short on time) ---
SEEDS = '42,52,62,72,82'
EPOCHS = 500
PATIENCE = 50
TRIALS = 12
ALLOC = 'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True'

def phase_done(table_name):
    return os.path.exists(f'{OUTDIR}/bot/tables/{table_name}.csv')

def run(cmd):
    print('\n>>>', cmd, flush=True)
    code = os.system(cmd)
    if code != 0:
        raise RuntimeError(f'Command failed (exit {code}): {cmd}')

base = f'{ALLOC} python src/run_experiments.py --task bot --epochs {EPOCHS} --patience {PATIENCE} --output-dir "{OUTDIR}" --quiet'

# Phase 1: all baselines + significance tests (skips if already done)
if phase_done('baseline_comparison'):
    print('[skip] baselines already complete')
else:
    run(f'{base} --mode baselines --seeds {SEEDS}')

# Phase 2: ablation study
if phase_done('ablation_comparison'):
    print('[skip] ablation already complete')
else:
    run(f'{base} --mode ablation --seeds {SEEDS}')

# Phase 3: hyperparameter tuning
if phase_done('hyperparameter_tuning'):
    print('[skip] tuning already complete')
else:
    run(f'{base} --mode tune --trials {TRIALS}')

print('\nAll phases complete. Tables are in', f'{OUTDIR}/bot/tables')

## 6. Preview the key paper tables

In [ ]:
import pandas as pd, os
OUTDIR = '/content/drive/MyDrive/HMR-GNN-results'
def show(path, title):
    if os.path.exists(path):
        print('\n===', title, '===')
        display(pd.read_csv(path))
    else:
        print('(missing)', path)
show(f'{OUTDIR}/bot/tables/baseline_comparison.csv', 'BASELINE COMPARISON (mean +/- std)')
show(f'{OUTDIR}/bot/tables/ablation_comparison.csv', 'ABLATION STUDY')
show(f'{OUTDIR}/bot/tables/significance_tests.csv', 'SIGNIFICANCE TESTS')
show(f'{OUTDIR}/bot/tables/hyperparameter_tuning.csv', 'HYPERPARAMETER TUNING')

## 7. Download all results as a zip
Your results are already saved in Google Drive (`MyDrive/HMR-GNN-results`), so they are
safe even if the session ends. This cell also bundles them into a single zip to download.

In [ ]:
import shutil
OUTDIR = '/content/drive/MyDrive/HMR-GNN-results'
shutil.make_archive('/content/hmrgnn_results', 'zip', OUTDIR)
from google.colab import files
files.download('/content/hmrgnn_results.zip')
print('Download started: hmrgnn_results.zip')